In [7]:
import pandas as pd

# Read dataset
data = pd.read_csv("data set.csv")

# Remove RollNo column if it exists
if "RollNo" in data.columns:
    data = data.drop(columns=["RollNo"])

# Shuffle the dataset
data = data.sample(frac=1, random_state=1).reset_index(drop=True)

# Split into training and testing data
split = int(len(data) * 0.8)
train = data[:split]
test = data[split:].copy()      # Create a copy to avoid SettingWithCopyWarning

# Separate features and target
target = data.columns[-1]
features = data.columns[:-1]

# Calculate prior probabilities
def prior(train):
    counts = train[target].value_counts()
    return (counts / len(train)).to_dict()

# Calculate likelihood with Laplace smoothing
def likelihood(train, feature, value, label):
    class_data = train[train[target] == label]
    match = len(class_data[class_data[feature] == value])
    return (match + 1) / (len(class_data) + train[feature].nunique())

# Predict the class of one sample
def predict(sample):
    priors = prior(train)
    probs = {}

    for label in priors:
        p = priors[label]

        for feature in features:
            p *= likelihood(train, feature, sample[feature], label)

        probs[label] = p

    return max(probs, key=probs.get)

# Predict all test samples
predictions = []

for _, row in test.iterrows():
    predictions.append(predict(row))

# Add predictions to the test dataset
test["Predicted"] = predictions

# Display the output
print(test)

            Age  Income Student Credit Rating Buys Computer Predicted
11        Young     Low     Yes          Fair           Yes       Yes
12  Middle Aged  Medium      No     Excellent           Yes       Yes
13       Senior     Low     Yes     Excellent            No       Yes
